---



# Unit 2 Assignment: Mixture of Experts (MoE) Router

## 1. Objective

Build a **Smart Customer Support Router** that sends user requests to the right expert:
- **Technical Expert**: bugs, coding, debugging
- **Billing Expert**: charges, refunds, invoices
- **General Expert**: fallback for normal chat
- **Tool Expert (Bonus)**: calls a mock tool for real-time-like data (Bitcoin price)

We will use **Groq API** with different **system prompts** to simulate domain experts.

## 2. Architecture (Router + Experts)

```mermaid
graph TD
    U[User Query] --> R[route_prompt(user_input)]
    R -->|technical| T[Technical Expert Prompt]
    R -->|billing| B[Billing Expert Prompt]
    R -->|general| G[General Expert Prompt]
    R -->|tool| X[Tool Expert Function]
    T --> O[Final Response]
    B --> O
    G --> O
    X --> O
```

In [1]:
# Setup
%pip install groq python-dotenv --upgrade --quiet

from dotenv import load_dotenv
load_dotenv()

import os
import getpass
import json
import time
from groq import Groq
from datetime import datetime

if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API Key: ")

client = Groq(api_key=os.environ["GROQ_API_KEY"])
ROUTER_MODELS = ["llama-3.1-8b-instant", "llama-3.3-70b-versatile"]
EXPERT_MODELS = ["llama-3.3-70b-versatile", "llama-3.1-8b-instant"]

# Project metadata
PROJECT_NAME = "Mixture of Experts (MoE) Customer Support Router"
PROJECT_VERSION = "1.0"
EXECUTION_TIMESTAMP = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

print(f"\n{'='*70}")
print(f"Project: {PROJECT_NAME}")
print(f"Version: {PROJECT_VERSION}")
print(f"Execution Time: {EXECUTION_TIMESTAMP}")
print(f"{'='*70}\n")

def invoke_with_fallback(messages, temperature, model_candidates):
    last_error = None
    for model_name in model_candidates:
        try:
            completion = client.chat.completions.create(
                model=model_name,
                temperature=temperature,
                messages=messages,
            )
            return completion
        except Exception as e:
            last_error = e
            # Try next model when one is deprecated/decommissioned.
            if "decommissioned" in str(e).lower() or "no longer supported" in str(e).lower():
                continue
            continue

    raise RuntimeError(f"All candidate models failed. Last error: {last_error}")

print("✓ Setup complete: API client initialized with fallback model strategy")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 5.5 MB/s eta 0:00:00
Enter your Groq API Key: ··········

Project: Mixture of Experts (MoE) Customer Support Router
Version: 1.0
Execution Time: 2026-03-31 16:04:00

✓ Setup complete: API client initialized with fallback model strategy


## 3. Define Experts (MODEL_CONFIG)

Each expert uses the same base model but a different **system role**.

In [2]:
MODEL_CONFIG = {
    "technical": {
        "models": EXPERT_MODELS,
        "temperature": 0.7,
        "system_prompt": (
            "You are a Technical Support Expert with 10+ years of experience. Your expertise includes: "
            "• Debugging code across Python, JavaScript, Java, C++\n"
            "• Diagnosing system errors, performance issues, and architecture problems\n"
            "• Explaining complex technical concepts in clear, accessible terms\n"
            "• Providing reproducible code examples and step-by-step solutions\n"
            "When responding: (1) Identify root cause, (2) Explain why, (3) Provide actionable steps, "
            "(4) Include code examples when relevant. Be precise, structured, and solution-focused."
        ),
    },
    "billing": {
        "models": EXPERT_MODELS,
        "temperature": 0.7,
        "system_prompt": (
            "You are a Billing & Account Support Specialist. Your expertise includes: "
            "• Payment processing, invoicing, and account management\n"
            "• Refund policies, dispute resolution, and credit adjustments\n"
            "• Subscription management, billing cycles, and pricing tiers\n"
            "• Understanding regulatory compliance and industry standards\n"
            "When responding: (1) Show empathy, (2) Gather relevant details, (3) Explain policies clearly, "
            "(4) Offer concrete next steps. Be transparent and customer-focused."
        ),
    },
    "general": {
        "models": EXPERT_MODELS,
        "temperature": 0.7,
        "system_prompt": (
            "You are a Professional Customer Support Representative. Your expertise includes: "
            "• Product knowledge across all offerings\n"
            "• Best practices in customer service and communication\n"
            "• Empathetic listening and problem-solving\n"
            "• Escalation procedures and resource identification\n"
            "When responding: (1) Acknowledge the question, (2) Provide clear information, "
            "(3) Offer relevant resources, (4) Provide follow-up options. Be helpful, professional, and outcome-focused."
        ),
    },
    "tool": {
        "models": ROUTER_MODELS,
        "temperature": 0.0,
        "system_prompt": "You are an intelligent tool-routing assistant. Identify when real-time data or external tools are needed."
    },
}

print("✓ MODEL_CONFIG initialized with 4 specialized expert configurations")
print(f"✓ Experts: {', '.join(MODEL_CONFIG.keys())}")

✓ MODEL_CONFIG initialized with 4 specialized expert configurations
✓ Experts: technical, billing, general, tool


## 4. Router Function (Core Task)

Constraint: return only one label from:
`technical`, `billing`, `general`, `tool`

In [3]:
def route_prompt(user_input: str) -> str:
    """Route user input to the appropriate expert based on intent classification.

    Args:
        user_input: The user's query or request

    Returns:
        One of: 'technical', 'billing', 'general', 'tool'

    Design: Uses temperature=0.0 for deterministic, consistent routing
    """
    routing_instruction = (
        "You are an intelligent request router. Classify the user input into exactly ONE category:\n"
        "- 'technical': bugs, errors, coding, debugging, system issues, performance problems\n"
        "- 'billing': payments, charges, refunds, invoices, subscriptions, account credits\n"
        "- 'tool': requests for current/live market data (bitcoin price, stock prices, weather, etc.)\n"
        "- 'general': product info, account features, general questions, fallback category\n"
        "Return ONLY the category name (one word) and nothing else. Be strict and decisive."
    )

    completion = invoke_with_fallback(
        messages=[
            {"role": "system", "content": routing_instruction},
            {"role": "user", "content": user_input},
        ],
        temperature=0.0,
        model_candidates=ROUTER_MODELS,
    )

    category = completion.choices[0].message.content.strip().lower()

    # Guardrail: enforce valid output even if model drifts
    valid_categories = {"technical", "billing", "general", "tool"}
    if category not in valid_categories:
        return "general"
    return category

print("✓ route_prompt() function defined - Intent classification ready")

✓ route_prompt() function defined - Intent classification ready


## 5. Bonus Tool Function

A mock function that simulates a tool call for Bitcoin price queries.

In [4]:
def mock_get_bitcoin_price() -> str:
    """Simulate a real-time Bitcoin price lookup.

    In production, this would call an external API like CoinGecko or Binance.
    For demonstration, we return a realistic mock value.
    """
    mock_price_usd = 68421.42
    market_cap = "1.34 trillion"
    change_24h = "+2.35%"
    return (
        f"\n📊 **Bitcoin Market Data (Live)**\n"
        f"Price: ${mock_price_usd:,.2f} USD\n"
        f"24h Change: {change_24h}\n"
        f"Market Cap: {market_cap}\n"
        f"Data Source: Mock API (Demo)\n"
    )

print("✓ mock_get_bitcoin_price() function defined - Tool routing ready")

✓ mock_get_bitcoin_price() function defined - Tool routing ready


## 6. Orchestrator Function

This function runs the full MoE flow:
1. Route user query
2. Pick expert config
3. Call tool or LLM expert
4. Return final answer

In [5]:
def process_request(user_input: str) -> dict:
    """Orchestrate the full MoE flow: route -> expert selection -> response.

    Args:
        user_input: The user's query or request

    Returns:
        Dictionary with 'category', 'response', and execution metadata
    """
    start_time = time.time()

    # Step 1: Route to appropriate expert
    category = route_prompt(user_input)

    # Step 2: Handle tool requests separately
    if category == "tool":
        tool_result = mock_get_bitcoin_price()
        execution_time = time.time() - start_time
        return {
            "category": category,
            "response": tool_result,
            "execution_time_ms": round(execution_time * 1000, 2),
            "model_used": "Tool/API"
        }

    # Step 3: Select expert configuration
    expert = MODEL_CONFIG.get(category, MODEL_CONFIG["general"])

    # Step 4: Invoke expert LLM
    completion = invoke_with_fallback(
        messages=[
            {"role": "system", "content": expert["system_prompt"]},
            {"role": "user", "content": user_input},
        ],
        temperature=expert["temperature"],
        model_candidates=expert["models"],
    )

    answer = completion.choices[0].message.content.strip()
    execution_time = time.time() - start_time

    return {
        "category": category,
        "response": answer,
        "execution_time_ms": round(execution_time * 1000, 2),
        "model_used": completion.model
    }

print("✓ process_request() orchestrator function defined - Ready for execution")

✓ process_request() orchestrator function defined - Ready for execution


## 7. Test Cases

Lets validate routing + expert behavior with assignment-style examples.

In [6]:
# Comprehensive test suite with diverse scenarios
tests = [
    {"query": "My Python script is throwing an IndexError on line 5. How do I fix it?", "expected": "technical"},
    {"query": "I was charged twice for my subscription this month - please help!", "expected": "billing"},
    {"query": "How can I improve my productivity while studying for exams?", "expected": "general"},
    {"query": "What is the current price of Bitcoin right now?", "expected": "tool"},
    {"query": "My payment failed but my account shows pending - what should I do?", "expected": "billing"},
    {"query": "Can you help me understand the error stacktrace in my Java application?", "expected": "technical"},
    {"query": "Tell me about your product features and pricing tiers.", "expected": "general"},
]

print("\n" + "="*70)
print("🚀 EXECUTING TEST CASES - MoE ROUTER VALIDATION")
print("="*70 + "\n")

results = []
execution_start = time.time()

for idx, test_case in enumerate(tests, 1):
    query = test_case["query"]
    expected = test_case["expected"]

    result = process_request(query)
    actual = result["category"]
    is_correct = actual == expected

    results.append({
        "test_num": idx,
        "query": query,
        "expected": expected,
        "actual": actual,
        "correct": is_correct,
        "execution_time_ms": result["execution_time_ms"]
    })

    # Display result
    status = "✓ PASS" if is_correct else "✗ FAIL"
    print(f"Test {idx}: {status}")
    print(f"  Query: {query[:70]}..." if len(query) > 70 else f"  Query: {query}")
    print(f"  Expected: {expected:12} | Actual: {actual:12}")
    print(f"  Response: {result['response'][:150]}..." if len(result['response']) > 150 else f"  Response: {result['response']}")
    print(f"  Time: {result['execution_time_ms']}ms | Model: {result['model_used']}")
    print()


🚀 EXECUTING TEST CASES - MoE ROUTER VALIDATION

Test 1: ✓ PASS
  Query: My Python script is throwing an IndexError on line 5. How do I fix it?
  Expected: technical    | Actual: technical   
  Response: # Step-by-step analysis of the problem:
1. **Understanding the Error**: An `IndexError` in Python occurs when you try to access an element in a list o...
  Time: 2397.11ms | Model: llama-3.3-70b-versatile

Test 2: ✓ PASS
  Query: I was charged twice for my subscription this month - please help!
  Expected: billing      | Actual: billing     
  Response: I'm so sorry to hear that you've been charged twice for your subscription this month. I can imagine how frustrating that must be for you. 

To help me...
  Time: 789.75ms | Model: llama-3.3-70b-versatile

Test 3: ✓ PASS
  Query: How can I improve my productivity while studying for exams?
  Expected: general      | Actual: general     
  Response: I understand that you're looking for ways to boost your productivity while studying for exa

In [7]:
# Analyze test results
total_tests = len(results)
passed_tests = sum(1 for r in results if r["correct"])
failed_tests = total_tests - passed_tests
accuracy = (passed_tests / total_tests * 100) if total_tests > 0 else 0
avg_execution_time = sum(r["execution_time_ms"] for r in results) / total_tests if total_tests > 0 else 0

print("\n" + "="*70)
print("📊 TEST RESULTS SUMMARY")
print("="*70)
print(f"Total Tests Run: {total_tests}")
print(f"Passed: {passed_tests} ✓")
print(f"Failed: {failed_tests} ✗")
print(f"Success Rate: {accuracy:.1f}%")
print(f"Average Execution Time: {avg_execution_time:.2f}ms")
print(f"Total Execution Time: {(time.time() - execution_start):.2f}s")
print("="*70 + "\n")

# Display detailed results in table format
if results:
    print("Detailed Results Table:")
    print(f"{'Test':<6} {'Expected':<12} {'Actual':<12} {'Status':<10} {'Query Preview':<45}")
    print("-" * 85)
    for r in results:
        status = "✓ PASS" if r["correct"] else "✗ FAIL"
        query_preview = r["query"][:42] + "..." if len(r["query"]) > 42 else r["query"]
        print(f"{r['test_num']:<6} {r['expected']:<12} {r['actual']:<12} {status:<10} {query_preview:<45}")

print("\n")


📊 TEST RESULTS SUMMARY
Total Tests Run: 7
Passed: 7 ✓
Failed: 0 ✗
Success Rate: 100.0%
Average Execution Time: 1434.05ms
Total Execution Time: 16.06s

Detailed Results Table:
Test   Expected     Actual       Status     Query Preview                                
-------------------------------------------------------------------------------------
1      technical    technical    ✓ PASS     My Python script is throwing an IndexError...
2      billing      billing      ✓ PASS     I was charged twice for my subscription th...
3      general      general      ✓ PASS     How can I improve my productivity while st...
4      tool         tool         ✓ PASS     What is the current price of Bitcoin right...
5      billing      billing      ✓ PASS     My payment failed but my account shows pen...
6      technical    technical    ✓ PASS     Can you help me understand the error stack...
7      general      general      ✓ PASS     Tell me about your product features and pr...




## 8. Results Summary & Evaluation

## 9. Architecture & Design Patterns

### Pattern: Mixture of Experts (MoE)
This implementation demonstrates a **router-based expert system** architecture:

**Components:**
1. **Router** (`route_prompt`): Intent classifier with temperature=0 for deterministic routing
2. **Experts** (`MODEL_CONFIG`): Specialized system prompts for different domains
3. **Orchestrator** (`process_request`): Combines routing + expert invocation
4. **Tool Layer**: Direct function calls for real-time data (Bitcoin price)

**Advantages:**
-  Specialized expertise per domain → Better quality responses
-  Deterministic routing → Consistent behavior
-  Scalable → Easy to add new experts
-  Fallback mechanisms → Robust error handling
-  Execution monitoring → Performance tracking

**Production Considerations:**
- Add caching for repeated queries
- Implement rate limiting and quota management
- Log all routing decisions for analytics
- A/B test expert prompts for continuous improvement
- Add multi-turn conversation state management

## 10. What You Built

You implemented a production-ready **Mixture of Experts (MoE) Router** with:

 **Intent Classification** - `route_prompt()` deterministically routes queries  
 **Expert Configuration** - 4 specialized experts with custom system prompts  
 **Smart Orchestration** - `process_request()` manages routing + expert invocation  
 **Tool Integration** - Separate pathway for real-time data queries  
 **Performance Monitoring** - Execution time tracking and performance metrics  
 **Comprehensive Testing** - 7 diverse test cases validating all expert paths  
 **Error Resilience** - Fallback models and guardrails  

### Real-World Applications:
- **Customer Support Systems** - Route tickets to right department
- **LLM API Gateways** - Intelligent request distribution
- **Search & Ranking** - Query classification and specialization
- **Content Recommendation** - Intent-based personalization
- **Healthcare/Finance** - Domain-specific expertise routing

This is a core architecture pattern used in production AI systems by companies like OpenAI, Anthropic, and major enterprises.
